# Chapter 39: Clustering and Segmentation

Synthetic NRG distributor behaviour illustrates geometry, validation, profiling, and stability.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans,AgglomerativeClustering,DBSCAN
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.clustering import fit_scaler,apply_scaler,fit_clusters,cluster_profile,silhouette_report,partition_agreement
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(39);sizes=[90,75,65]
centres=np.array([[12,2.5,.15],[4,8,.55],[8,5,.30]])
parts=[rng.normal(c,[1.7,.7,.05],size=(n,3)) for c,n in zip(centres,sizes)]
x=np.vstack(parts);feature_names=['orders_per_month','product_families','return_rate']
center,scale=fit_scaler(x);z=apply_scaler(x,center,scale)
print(f'Distributors: {len(x)}; features: {x.shape[1]}')


Distributors: 230; features: 3


In [ ]:
rows=[]
for k in range(2,7):
 model,labels=fit_clusters(KMeans(n_clusters=k,n_init=20),z);report=silhouette_report(z,labels);rows.append((k,model.inertia_,report['mean'],report['negative_count']))
for row in rows:print(f'k={row[0]} inertia={row[1]:.1f} silhouette={row[2]:.3f} negatives={row[3]}')


k=2 inertia=185.2 silhouette=0.616 negatives=1
k=3 inertia=79.1 silhouette=0.597 negatives=0
k=4 inertia=67.7 silhouette=0.491 negatives=1
k=5 inertia=58.4 silhouette=0.412 negatives=0
k=6 inertia=49.7 silhouette=0.322 negatives=1


In [ ]:
model,labels=fit_clusters(KMeans(n_clusters=3,n_init=30),z);profile=cluster_profile(x,labels)
for k,v in profile.items():
 summary={name:round(float(value),2) for name,value in zip(feature_names,v['mean'])};print(k,v['count'],summary)
_,labels_2=fit_clusters(KMeans(n_clusters=3,n_init=30),z,seed=99);print(f'Seed agreement: {partition_agreement(labels,labels_2):.3f}')


0 75 {'orders_per_month': 3.76, 'product_families': 8.13, 'return_rate': 0.55}
1 89 {'orders_per_month': 11.93, 'product_families': 2.64, 'return_rate': 0.15}
2 66 {'orders_per_month': 7.72, 'product_families': 5.01, 'return_rate': 0.29}
Seed agreement: 1.000


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot([r[0] for r in rows],[r[1] for r in rows],'o-');axes[0].set(xlabel='k',ylabel='Inertia',title='Compactness')
scatter=axes[1].scatter(z[:,0],z[:,1],c=labels,cmap='viridis',s=22);axes[1].set(xlabel='Scaled order frequency',ylabel='Scaled product breadth',title='Selected partition')
fig.tight_layout();plt.show()


## Interpretation

Internal geometry supports three compact groups in this synthetic example. A production choice must also survive time and sampling changes, produce viable segment sizes, pass stakeholder review, and support a measurable action.


In [ ]:
# Practice: compare hierarchical clustering and DBSCAN on the same scaled representation.
